In [1]:
import pandas as pd
import requests
from io import BytesIO
from PyPDF2 import PdfReader
import re

# ===== 1. Load CSV =====
df = pd.read_csv("/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/data/merged_ntsb_data.csv", sep=None, engine='python', encoding='utf-8-sig')  
df = df.dropna(subset=["ReportUrl"]).reset_index(drop=True)

# Only use first line for now
row = df.iloc[0]
print(row)
pdf_url = row["ReportUrl"]
print(pdf_url)

# ===== 2. Download PDF =====
response = requests.get(pdf_url)
pdf_bytes = BytesIO(response.content)

# ===== 3. Extract text =====
reader = PdfReader(pdf_bytes)
full_text = "\n".join(page.extract_text() for page in reader.pages)
print(full_text)

# ===== 4. Parse into structured components =====
def extract_section(text, header, next_header_list):
    """Extract a section by header until next header."""
    pattern = re.escape(header) + r"(.*?)(?=" + "|".join(map(re.escape, next_header_list)) + r"|$)"
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

sections = {
    "BasicInfo": extract_section(full_text, "Aviation Investigation Final Report", ["Analysis"]),
    "Analysis": extract_section(full_text, "Analysis", ["Probable Cause", "Findings", "Factual Information"]),
    "ProbableCause": extract_section(full_text, "Probable Cause", ["Findings", "Factual Information"]),
    "Findings": extract_section(full_text, "Findings", ["Factual Information", "Pilot Information"]),
    "FactualInfo": extract_section(full_text, "Factual Information", ["Pilot Information", "Aircraft and Owner/Operator"]),
    "PilotInfo": extract_section(full_text, "Pilot Information", ["Aircraft and Owner/Operator"]),
    "AircraftInfo": extract_section(full_text, "Aircraft and Owner/Operator Information", ["Meteorological", "Administrative"]),
    "WeatherInfo": extract_section(full_text, "Meteorological Information", ["Administrative", "Note:"]),
    "AdministrativeInfo": extract_section(full_text, "Administrative Information", ["Note:"]),
}

# ===== 5. Add to DataFrame =====
for k, v in sections.items():
    df.loc[0, k] = v

# ===== 6. Save output to new CSV =====
df.to_csv("ntsb_with_sections.csv", index=False)

print("Extracted sections:")
for k in sections:
    print(f"{k}: {len(sections[k])} chars")


NtsbNo                                                                ANC26LA008
EventType                                                                    ACC
Mkey                                                                      202156
EventDate                                                   2025-12-12T18:01:00Z
City                                                                      Palmer
State                                                                     Alaska
Country                                                            United States
ReportNo                                                                     NaN
N#                                                                        N20305
SerialNumber                                                            17261176
HasSafetyRec                                                               False
Mode                                                                    Aviation
ReportType                  

In [2]:
import pandas as pd
import requests
from io import BytesIO
from PyPDF2 import PdfReader
from tqdm import tqdm
import time

# ===== 1. Load CSV =====
df = pd.read_csv(
    "/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/data/merged_ntsb_data.csv",
    sep=None, engine='python', encoding='utf-8-sig'
)
df = df.dropna(subset=["ReportUrl"]).reset_index(drop=True)
print(f"Loaded {len(df)} valid report URLs")

# ===== 2. Helper: download + extract text =====
def fetch_pdf_text(url, retries=2, delay=3):
    """Download PDF and extract its text, with retries."""
    if not isinstance(url, str) or not url.strip():
        return None

    for attempt in range(retries + 1):
        try:
            response = requests.get(url.strip(), timeout=20)
            response.raise_for_status()
            reader = PdfReader(BytesIO(response.content))
            return "\n".join(page.extract_text() or "" for page in reader.pages)
        except Exception as e:
            if attempt < retries:
                time.sleep(delay)
            else:
                print(f"⚠️ Failed to process {url}: {e}")
                return None

# ===== 3. Extract PDFs for all rows =====
texts = []
for url in tqdm(df["ReportUrl"], desc="Extracting PDFs"):
    text = fetch_pdf_text(url)
    texts.append(text)

df["fulltext"] = texts

# ===== 4. Save to new CSV =====
out_path = "ntsb_with_fulltext.csv"
df.to_csv(out_path, index=False)
print(f"\n✅ Done! Saved {len(df)} rows to {out_path}")


Loaded 1890 valid report URLs


Extracting PDFs:  40%|████      | 765/1890 [44:13<1:46:43,  5.69s/it]

⚠️ Failed to process https://data.ntsb.gov/carol-repgen/api/Aviation/ReportMain/GenerateNewestReport/106951/pdf: 403 Client Error: Forbidden for url: https://data.ntsb.gov/carol-repgen/api/Aviation/ReportMain/GenerateNewestReport/106951/pdf


Extracting PDFs:  41%|████      | 766/1890 [44:20<1:51:18,  5.94s/it]

⚠️ Failed to process https://data.ntsb.gov/carol-repgen/api/Aviation/ReportMain/GenerateNewestReport/106933/pdf: 403 Client Error: Forbidden for url: https://data.ntsb.gov/carol-repgen/api/Aviation/ReportMain/GenerateNewestReport/106933/pdf


...

⚠️ Failed to process https://www.ntsb.gov/investigations/Pages/DCA20MA059.aspx: EOF marker not found


Extracting PDFs:  94%|█████████▍| 1784/1890 [1:47:13<07:57,  4.51s/it]  

⚠️ Failed to process https://www.ntsb.gov/investigations/Pages/DCA24MA063.aspx: EOF marker not found


Extracting PDFs: 100%|██████████| 1890/1890 [1:54:22<00:00,  3.63s/it]


✅ Done! Saved 1890 rows to ntsb_with_fulltext.csv
